# This part will load and prepare data - explanation in code_example.ipynb

In [1]:
from load_dataset import OSAData
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
import torch.nn as nn
from tqdm import tqdm
import torch.optim as optim
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix,classification_report



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print('GPU')


GPU


In [2]:
load = OSAData("/content/data")
data=load.load_data()


In [3]:
WINDOW_SIZE = 60   

def find_event(t_win_start,t_win_end,events,margin=30):
    for _,ev_start,ev_end in events:
        if t_win_start < ev_end + margin and t_win_end > ev_start - margin:
            return True
    return False


apnea_events=[]
normal_signals=[]
signals=[]
labels=[]


mapping = {
        'osa': 1,
        'hypo': 1
    }

for i in range(len(data)): 
    data_num=data[i]
    single_event=[]

    for event in data_num['annotation']:
        apnea_events.append([event['event_type'],event['evnet_start'],event['evnet_start']+int(event['event_duration'])])
        single_event.append([event['event_type'],event['evnet_start'],event['evnet_start']+int(event['event_duration'])])

    spo2_val=data_num['spo2_values']
    spo2_t=data_num['spo2_time']

    for event,t_start,t_end in single_event:
        mask = (data_num['spo2_time'] >= t_start) & (data_num['spo2_time'] <= t_start + 59)
        event_spo2 = data_num['spo2_values'][mask]
        if len(event_spo2) == 60:
            labels.append(event)
            signals.append(event_spo2)

            
    t_begining=spo2_t[0]+30
    t_end=t_begining+60

    while t_end<spo2_t[-1]:
        if find_event(t_begining,t_end,single_event) == False:
            mask = (data_num['spo2_time'] >= t_begining) & (data_num['spo2_time'] < t_end)
            event_spo2 = data_num['spo2_values'][mask]
            if len(event_spo2)==WINDOW_SIZE:
                normal_signals.append(event_spo2)
        
        t_begining+=WINDOW_SIZE
        t_end+=WINDOW_SIZE




In [4]:
X = np.array(signals + normal_signals)
y = np.array([mapping[label] for label in labels] + [0] * len(normal_signals))

X = (X - X.mean()) / X.std()


In [5]:
X_train, X_temp, y_train, y_temp = train_test_split(X,y,test_size=0.2,random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp,test_size=0.5,stratify=y_temp)

In [6]:
_,counts=np.unique(y_train,return_counts=True)
sample_weights = [1/counts[i] for i in y_train]

sampler = WeightedRandomSampler(sample_weights, len(sample_weights))


In [7]:
X_train_tensor = torch.from_numpy(X_train).unsqueeze(-1)
y_train_tensor = torch.from_numpy(y_train).long()
X_test_tensor = torch.from_numpy(X_test).unsqueeze(-1)
y_test_tensor = torch.from_numpy(y_test).long()
X_val_tensor = torch.from_numpy(X_val).unsqueeze(-1)
y_val_tensor = torch.from_numpy(y_val).long()


In [8]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

batch_size = 32
num_classes = 3

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False, sampler=sampler)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [9]:
class CNNLSTMClassifier(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        self.cnn = nn.Sequential(

          
            nn.Conv1d(1, 32, kernel_size=11, padding=5),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),

          
            nn.Conv1d(32, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),

         
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.Dropout(0.3)
        )

        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=64,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.3
        )

        self.fc = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):

      
        x = x.permute(0, 2, 1)
        x = self.cnn(x)
        x = x.permute(0, 2, 1)
        out, _ = self.lstm(x)
        x = out[:, -1, :]

        return self.fc(x)

model = CNNLSTMClassifier().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [10]:
def train(model, loader):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for x, y in loader:
        x, y = x.float().to(device), y.to(device)
        optimizer.zero_grad()
        scores = model(x)
        loss = loss_fn(scores, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, predicted = torch.max(scores, 1)
        correct += (predicted == y).sum().item()
        total += y.size(0)

    avg_loss=total_loss / len(loader)


    model.eval()
    val_loss = 0.0

    with torch.no_grad():
            for val_data, val_label in val_loader:
                val_data = val_data.float().to(device)
                val_label = val_label.to(device)
                val_outputs = model(val_data)
                loss = loss_fn(val_outputs, val_label)
                val_loss += loss.item()

                
    avg_val_loss = val_loss / len(val_loader)

    return avg_loss, avg_val_loss, correct / total


loss_epoch=[]
loss_epoch_val=[]
epochs = 25
progress_bar = tqdm(range(epochs))
for epoch in progress_bar:
    loss,loss_val, acc = train(model, train_loader)
    print(f'Epoch [{epoch+1}/{epochs}] | Loss: {loss} | Accuracy: {acc}')
    loss_epoch.append(loss)
    loss_epoch_val.append(loss_val)




  4%|▍         | 1/25 [00:02<01:08,  2.86s/it]

Epoch [1/25] | Loss: 0.44118969369088945 | Accuracy: 0.8103675205021768


  8%|▊         | 2/25 [00:04<00:51,  2.25s/it]

Epoch [2/25] | Loss: 0.3866534191138536 | Accuracy: 0.8467145894502379


 12%|█▏        | 3/25 [00:06<00:45,  2.06s/it]

...

Epoch [23/25] | Loss: 0.27569594812914006 | Accuracy: 0.8876176976814822


 96%|█████████▌| 24/25 [00:46<00:01,  1.87s/it]

Epoch [24/25] | Loss: 0.2885642204325176 | Accuracy: 0.8822516958590665


100%|██████████| 25/25 [00:48<00:00,  1.94s/it]

Epoch [25/25] | Loss: 0.2864687642908405 | Accuracy: 0.8841753568897438


In [11]:
def evaluate(model, loader):
    model.to(device)
    all_y, all_pred = [], []
    model.eval()
    with torch.no_grad():
        for x, y in loader:
          x=x.float().to(device)
          y=y.to(device)
          y_pred = model(x)
          _, predicted = torch.max(y_pred.data, 1)
          all_y.append(y.cpu().numpy())
          all_pred.append(predicted.cpu().numpy())


    return np.concatenate(all_y), np.concatenate(all_pred)


y_true, y_pred = evaluate(model, test_loader)

cm=confusion_matrix(y_true, y_pred)
cr=classification_report(y_true, y_pred)
print('Macierz błędów:\n')
print(cm)
print('\n\nRaport klasyfikacji:\n')
print(cr)


Macierz błędów:

[[598  57]
 [ 95 485]]


Raport klasyfikacji:

              precision    recall  f1-score   support

           0       0.86      0.91      0.89       655
           1       0.89      0.84      0.86       580

    accuracy                           0.88      1235
   macro avg       0.88      0.87      0.88      1235
weighted avg       0.88      0.88      0.88      1235

